In [ ]:

!pip install datasets pandas numpy scikit-learn imbalanced-learn joblib
!pip install xgboost

import pandas as pd
import numpy as np
import joblib

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
!pip install datasets pandas numpy scikit-learn imbalanced-learn joblib

import pandas as pd
import numpy as np
import joblib

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
ds = load_dataset("electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset")

df = ds["train"].to_pandas()

print("Dataset shape:", df.shape)
print(df.head())

In [ ]:

df["is_fraud"] = df["is_fraud"].astype(int)

df = df.select_dtypes(include=[np.number])

df = df.dropna()

X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]

print("Final feature shape:", X.shape)
print("Fraud distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
log_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

log_model.fit(X_train, y_train)

y_proba_log = log_model.predict_proba(X_test)[:,1]
print("Logistic ROC-AUC:", roc_auc_score(y_test, y_proba_log))

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train),
    n_jobs=-1,
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_proba_xgb = xgb_model.predict_proba(X_test)[:,1]

print("XGBoost ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))

In [ ]:
from sklearn.metrics import roc_auc_score

print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, y_proba_log))
print("SVM ROC-AUC:", roc_auc_score(y_test, y_proba_svm))
print("XGBoost ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))

In [ ]:
final_model = svm_model
y_proba = y_proba_svm

In [ ]:
risk_score = y_proba * 100

In [ ]:
def risk_tier(score):
    if score < 30:
        return "LOW"
    elif score < 70:
        return "MEDIUM"
    else:
        return "HIGH"

risk_categories = [risk_tier(s) for s in risk_score]

In [ ]:
y_pred = final_model.predict(X_test)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
average_fraud_loss = 75000

fraud_detected = sum((y_test == 1) & (y_pred == 1))
estimated_savings = fraud_detected * average_fraud_loss

print("Fraud Cases Detected:", fraud_detected)
print("Estimated Fraud Prevented (NGN):", estimated_savings)

In [ ]:
importances = rf_model.feature_importances_

feature_rank = sorted(
    zip(X.columns, importances),
    key=lambda x: x[1],
    reverse=True
)

print("Top 10 Risk Drivers:")
for feature, score in feature_rank[:10]:
    print(feature, round(score, 4))

In [ ]:
joblib.dump(final_model, "nigerian_bank_fraud_model.pkl")

In [ ]:
psi = np.sum((expected - actual) * np.log(expected / actual))